In [1]:
import pandas as pd
import re
from collections import Counter
import seaborn as sns

%load_ext autoreload
%autoreload 2
import src.data_utils as utils

In [2]:
baseline_df = pd.read_json('../data/baseline_2025-06-26/PMC008xxxxxx_noncomm.json')

In [3]:
baseline_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 185530 entries, 0 to 185529
Data columns (total 11 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   article-type    185530 non-null  object 
 1   language        92879 non-null   object 
 2   journal         185530 non-null  object 
 3   pmc-id          185530 non-null  object 
 4   pmid            155910 non-null  float64
 5   title           185523 non-null  object 
 6   country         160471 non-null  object 
 7   date            185530 non-null  object 
 8   abstract        185530 non-null  object 
 9   section_titles  170086 non-null  object 
 10  sections        185530 non-null  object 
dtypes: float64(1), object(10)
memory usage: 17.0+ MB


In [4]:
baseline_df.head()

,article-type,language,journal,pmc-id,pmid,title,country,date,abstract,section_titles,sections
0,correction,None,Diabetes Therapy,PMC8991299,35286609.0,Correction to: Overt Diabetes in Pregnancy,India,14-3-2022,,[Correction to: Diabetes Ther 10.1007/s13300-0...,[Correction to: Diabetes Ther 10.1007/s13300-0...
1,research-article,en,International Journal of Chronic Obstructive P...,PMC8749770,35027824.0,Effect of Smoking and Its Cessation on the Tra...,India,05-1-2022,Rationale Smoking is the primary cause of chro...,"[Introduction, Methods, Results, Discussion, C...",[Introduction Chronic Obstructive Pulmonary Di...
2,research-article,en,Journal of Clinical Laboratory Analysis,PMC8605134,34528313.0,Evaluation of fungal contamination and ochrato...,"Iran, Islamic Republic of",15-9-2021,Abstract Background Mycotoxins are secondary f...,"[INTRODUCTION, MATERIALS, RESULT, DISCUSSION, ...",[1 INTRODUCTION Coffee is a member of Coffea g...
3,review-article,en,International Journal of Clinical Pediatric De...,PMC8783216,NaN,The Epiphany of Post-COVID: A Watershed for Pe...,India,Nov-Dec-2021,A bstract Coronavirus disease-2019 (COVID-19) ...,"[I, E, T, S, D, C]",[I ntroduction Coronavirus disease-2019 (COVID...
4,research-article,None,Schizophrenia Research: Cognition,PMC8131976,34026572.0,Screening for cognitive impairment in schizoph...,Austria,12-5-2021,Background The Screen for Cognitive Impairment...,"[Introduction, Methods and materials, Results,...",[1 Introduction Cognitive impairment is consid...


In [5]:
#replace None with empty string to make my life easier
baseline_df['section_titles'] = baseline_df['section_titles'].apply(
    lambda x: [] if x == None else x
)

In [6]:
def replace_section_titles(section_titles):

    alt_titles = [[utils.get_alt_title(title) for title in secs] 
                    for secs in section_titles]
    alt_titles = [[title for title in sorted(secs) if not title == '']
                  for secs in alt_titles]

    return alt_titles
    


In [7]:
baseline_df['section_titles_cleaned'] = replace_section_titles(list(baseline_df['section_titles']))

In [8]:
for i in range(10, 20):
    print(f'before: {baseline_df['section_titles'].iloc[i]}')
    print(f'after: {baseline_df['section_titles_cleaned'].iloc[i]}')

before: ['Le polysulfate de pentosan est le principal traitement de la douleur vésicale associée à la cystite interstitielle', 'La maculopathie est associée à une utilisation prolongée du polysulfate de pentosan', 'La maculopathie causée par le polysulfate de pentosan peut s’apparenter à la dégénérescence maculaire liée à l’âge', 'La maladie maculaire peut poursuivre sa progression malgré l’arrêt du polysulfate de pentosan', 'Les patients exposés au polysulfate de pentosan qui rapportent des troubles de la vision devraient subir un dépistage ophtalmique']
after: []
before: ['INTRODUCTION', 'BACKGROUND', 'THE STUDY', 'RESULTS', 'DISCUSSION', 'CONCLUSIONS', 'CONFLICT OF INTEREST', 'AUTHOR CONTRIBUTIONS', 'ETHICAL APPROVAL AND PATIENT CONSENT\u202fFOR PUBLICATION\u202fSTATEMENT']
after: ['background', 'conclusion', 'discussion', 'introduction', 'results']
before: []
after: []
before: ['Introduction', 'Case Report', 'Discussion']
after: ['case report', 'discussion', 'introduction']
before:

In [9]:
# most frequent section titles overall
Counter(map(str, baseline_df['section_titles_cleaned']))

Counter({"['discussion', 'introduction', 'methods', 'results']": 37848,
         "['conclusion', 'discussion', 'introduction', 'methods', 'results']": 37380,
         '[]': 29591,
         "['conclusion', 'introduction']": 11186,
         "['case report', 'conclusion', 'discussion', 'introduction']": 6295,
         "['discussion', 'methods', 'results']": 6076,
         "['case report', 'discussion', 'introduction']": 5601,
         "['conclusion', 'introduction', 'methods']": 5045,
         "['conclusion', 'discussion', 'methods', 'results']": 4615,
         "['conclusion', 'experiment', 'introduction']": 3443,
         "['introduction']": 3132,
         "['conclusion']": 2083,
         "['conclusion', 'discussion', 'introduction', 'limitations', 'methods', 'results']": 2023,
         "['background', 'conclusion', 'discussion', 'methods', 'results']": 1951,
         "['introduction', 'methods']": 1524,
         "['conclusion', 'discussion', 'introduction', 'methods', 'results', 'summar

##### Article type: research article

In [ ]:
count = len( baseline_df[baseline_df['article-type'] == 'research-article'])
prop = count / len(baseline_df)
print(f'{count} research articles ({"%.2f" % (prop*100)}% of total count)')

119546 research articles (64.43% of total count)


In [10]:
Counter(map(str, baseline_df[baseline_df['article-type'] == 'research-article']['section_titles_cleaned'])).most_common(5)

[("['discussion', 'introduction', 'methods', 'results']", 36026),
 ("['conclusion', 'discussion', 'introduction', 'methods', 'results']", 34643),
 ("['discussion', 'methods', 'results']", 5657),
 ('[]', 5219),
 ("['conclusion', 'introduction', 'methods']", 4464)]

##### Article type: review article

In [27]:
count = len( baseline_df[baseline_df['article-type'] == 'review-article'])
prop = count / len(baseline_df)
print(f'{count} review articles ({"%.2f" % (prop*100)}% of total count)')


16340 review articles (8.81% of total count)


In [11]:
Counter(map(str, baseline_df[baseline_df['article-type'] == 'review-article']['section_titles_cleaned'])).most_common(5)

[("['conclusion', 'introduction']", 6280),
 ("['conclusion', 'discussion', 'introduction', 'methods', 'results']", 1688),
 ("['introduction']", 1502),
 ('[]', 964),
 ("['conclusion']", 870)]

##### Article type: case report

In [ ]:
count = len( baseline_df[baseline_df['article-type'] == 'case-report'])
prop = count / len(baseline_df)
print(f'{count} case reports ({"%.2f" % (prop*100)}% of total count)')

18783 case reports (10.1% of total count)


In [12]:
Counter(map(str, baseline_df[baseline_df['article-type'] == 'case-report']['section_titles_cleaned'])).most_common(5)

[("['case report', 'conclusion', 'discussion', 'introduction']", 6059),
 ("['case report', 'discussion', 'introduction']", 5341),
 ('[]', 1201),
 ("['case report', 'conclusion', 'diagnosis', 'discussion', 'introduction', 'outcome', 'treatment']",
  940),
 ("['case report', 'discussion']", 577)]

In [13]:
# proportion of all papers where no titles were found
len(baseline_df[baseline_df['section_titles'].apply(lambda x: x == [])]) / len(baseline_df)

0.08324260227456476

In [14]:
# proportion of research articles in batch
article_count = len(baseline_df[baseline_df['article-type'] == 'research-article']) 
article_count / len(baseline_df)

0.6443486228642268

In [15]:
# proportion of research articles where no titles were found
len(baseline_df[baseline_df['section_titles'].apply(lambda x: x == [])][baseline_df['article-type'] == 'research-article']) / article_count

/tmp/ipykernel_1431432/1322135802.py:2: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  len(baseline_df[baseline_df['section_titles'].apply(lambda x: x == [])][baseline_df['article-type'] == 'research-article']) / article_count


0.00964482291335553

In [16]:
Counter(baseline_df['article-type'])

Counter({'research-article': 119546,
         'case-report': 18783,
         'review-article': 16340,
         'letter': 5391,
         'abstract': 5157,
         'editorial': 4767,
         'brief-report': 4645,
         'other': 4090,
         'correction': 1575,
         'discussion': 1281,
         'article-commentary': 1218,
         'systematic-review': 708,
         'data-paper': 438,
         'news': 395,
         'retraction': 315,
         'reply': 213,
         'rapid-communication': 178,
         'meeting-report': 131,
         'report': 94,
         'book-review': 61,
         'methods-article': 53,
         'obituary': 45,
         'introduction': 39,
         'expression-of-concern': 26,
         'in-brief': 15,
         'product-review': 11,
         'addendum': 9,
         'announcement': 2,
         'oration': 2,
         ' research-article': 1,
         'protocol': 1})